In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb
import joblib

df = pd.read_csv("../data/historical_trains.csv")
df.shape

(2880, 18)

In [2]:
# Work on a copy so we don't touch the original df
model_df = df.copy()

# Columns we'll encode
categorical_cols = ['train_id', 'origin_station', 'destination_station', 
                     'day_of_week', 'weather', 'congestion_level']

encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    model_df[col + '_enc'] = le.fit_transform(model_df[col])
    encoders[col] = le  # save encoder so we can decode/reuse later in the API

model_df[[c + '_enc' for c in categorical_cols]].head()

,train_id_enc,origin_station_enc,destination_station_enc,day_of_week_enc,weather_enc,congestion_level_enc
0,0,3,1,1,0,0
1,0,1,3,1,0,1
2,0,2,0,1,0,2
3,0,0,2,1,0,2
4,1,3,1,1,0,0


In [3]:
feature_cols = [
    'train_id_enc', 'origin_station_enc', 'destination_station_enc',
    'day_of_week_enc', 'weather_enc', 'congestion_level_enc',
    'current_delay_min', 'distance_km', 'hour_of_day',
    'is_weekend', 'is_holiday'
]

X = model_df[feature_cols]
y = model_df['future_delay_min']

print(X.shape, y.shape)
X.head()

(2880, 11) (2880,)


,train_id_enc,origin_station_enc,destination_station_enc,day_of_week_enc,weather_enc,congestion_level_enc,current_delay_min,distance_km,hour_of_day,is_weekend,is_holiday
0,0,3,1,1,0,0,0.0,95,5,False,False
1,0,1,3,1,0,1,27.8,55,6,False,False
2,0,2,0,1,0,2,37.4,40,7,False,False
3,0,0,2,1,0,2,44.7,60,8,False,False
4,1,3,1,1,0,0,0.0,95,7,False,False


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)

Train size: (2304, 11)
Test size: (576, 11)


In [5]:
model = xgb.XGBRegressor(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.1,
    random_state=42
)

model.fit(X_train, y_train)
print("Model trained successfully!")

Model trained successfully!


In [6]:
y_pred = model.predict(X_test)

model_mae = mean_absolute_error(y_test, y_pred)
print(f"XGBoost Model MAE: {model_mae:.2f} minutes")
print(f"Baseline MAE was: 20.16 minutes")
print(f"Improvement: {20.16 - model_mae:.2f} minutes ({((20.16 - model_mae)/20.16)*100:.1f}% better)")

XGBoost Model MAE: 5.85 minutes
Baseline MAE was: 20.16 minutes
Improvement: 14.31 minutes (71.0% better)


In [7]:
importance_df = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

importance_df

,feature,importance
6,current_delay_min,0.621760
5,congestion_level_enc,0.114575
10,is_holiday,0.101991
4,weather_enc,0.060124
9,is_weekend,0.033600
0,train_id_enc,0.029616
8,hour_of_day,0.010805
2,destination_station_enc,0.010443
3,day_of_week_enc,0.006401
1,origin_station_enc,0.005548


In [8]:
joblib.dump(model, "../ml/train_delay_model.pkl")
joblib.dump(encoders, "../ml/label_encoders.pkl")
joblib.dump(feature_cols, "../ml/feature_cols.pkl")

print("Model and encoders saved!")

Model and encoders saved!
